# Scorecard from exported artifacts

This notebook reads exported artifacts created by the GLM and FFNN notebooks and produces:
- A compact GLM vs FFNN scorecard
- Hybrid attribution rows (FFNN incidence × GLM severity; GLM incidence × FFNN severity)
- A slide-ready CSV output

Required exported files (default location: `outputs_artifacts/`):
- `glm_test_exposure_scored.csv`
- `glm_test_claims_scored.csv`
- `ffnn_test_exposure_scored.csv`
- `ffnn_test_claims_scored.csv`

Date created: 2026-06-06

In [6]:
# Common setup
import os
import sys
import platform
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True)

def make_ohe_dense(drop=None):
    """Dense one-hot encoder compatible across sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop=drop)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, drop=drop)

def make_ohe_sparse(drop=None):
    """Sparse one-hot encoder compatible across sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, drop=drop)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True, drop=drop)

def env_fingerprint():
    import sklearn
    fp = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
    }
    try:
        import tensorflow as tf
        fp["tensorflow"] = tf.__version__
    except Exception:
        fp["tensorflow"] = None
    return fp


In [7]:
# --- 1) Where to read artifacts from ---
ARTIFACT_DIR = "outputs_artifacts"

GLM_EXPO_SCORED   = os.path.join(ARTIFACT_DIR, "glm_test_exposure_scored.csv")
GLM_CLAIMS_SCORED = os.path.join(ARTIFACT_DIR, "glm_test_claims_scored.csv")
FFNN_EXPO_SCORED  = os.path.join(ARTIFACT_DIR, "ffnn_test_exposure_scored.csv")
FFNN_CLAIMS_SCORED= os.path.join(ARTIFACT_DIR, "ffnn_test_claims_scored.csv")

paths = [GLM_EXPO_SCORED, GLM_CLAIMS_SCORED, FFNN_EXPO_SCORED, FFNN_CLAIMS_SCORED]
for p in paths:
    print(p, "exists?", os.path.exists(p))

META_JSON = os.path.join(ARTIFACT_DIR, "run_metadata.json")
print("run_metadata.json exists?", os.path.exists(META_JSON))


outputs_artifacts\glm_test_exposure_scored.csv exists? True
outputs_artifacts\glm_test_claims_scored.csv exists? True
outputs_artifacts\ffnn_test_exposure_scored.csv exists? True
outputs_artifacts\ffnn_test_claims_scored.csv exists? True
run_metadata.json exists? False


In [8]:
# --- 2) Load artifacts ---
glm_expo = pd.read_csv(GLM_EXPO_SCORED, parse_dates=["month_start"])
glm_clm  = pd.read_csv(GLM_CLAIMS_SCORED, parse_dates=["month_start"])
fnn_expo = pd.read_csv(FFNN_EXPO_SCORED, parse_dates=["month_start"])
fnn_clm  = pd.read_csv(FFNN_CLAIMS_SCORED, parse_dates=["month_start"])

print("GLM expo rows:", len(glm_expo), " GLM claims rows:", len(glm_clm))
print("FFNN expo rows:", len(fnn_expo), " FFNN claims rows:", len(fnn_clm))

req_expo_cols = ["life_id","month_start","exposure_months","onset","claim_id","p_hat","sev_hat","exp_cost"]
req_clm_cols  = ["claim_id","life_id","month_start","total_paid_ultimate","sev_hat"]

for c in req_expo_cols:
    assert c in glm_expo.columns, f"Missing in GLM expo: {c}"
    assert c in fnn_expo.columns, f"Missing in FFNN expo: {c}"
for c in req_clm_cols:
    assert c in glm_clm.columns, f"Missing in GLM claims: {c}"
    assert c in fnn_clm.columns, f"Missing in FFNN claims: {c}"

# Ensure both models used the same test window
assert glm_expo["month_start"].min() == fnn_expo["month_start"].min()
assert glm_expo["month_start"].max() == fnn_expo["month_start"].max()


GLM expo rows: 36000  GLM claims rows: 384
FFNN expo rows: 36000  FFNN claims rows: 384


In [9]:
# Compute scorecard
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, mean_absolute_error

def top_decile_capture(y_true, p_hat, frac=0.10):
    k = max(1, int(frac * len(p_hat)))
    idx = np.argsort(p_hat)[-k:]
    return float(y_true[idx].sum() / max(1, y_true.sum()))

# Incidence
y = glm_expo["onset"].values.astype(float)
p_glm = glm_expo["p_hat"].values.astype(float)
p_fnn = fnn_expo["p_hat"].values.astype(float)

# Severity
y_sev = glm_clm["total_paid_ultimate"].values.astype(float)
sev_glm = glm_clm["sev_hat"].values.astype(float)
sev_fnn = fnn_clm["sev_hat"].values.astype(float)

# Incidence metrics
inc = dict(
    observed_rate=float(y.mean()),
    glm_mean_p=float(p_glm.mean()),
    fnn_mean_p=float(p_fnn.mean()),
    glm_auc=float(roc_auc_score(y, p_glm)),
    fnn_auc=float(roc_auc_score(y, p_fnn)),
    glm_pr=float(average_precision_score(y, p_glm)),
    fnn_pr=float(average_precision_score(y, p_fnn)),
    glm_brier=float(brier_score_loss(y, p_glm)),
    fnn_brier=float(brier_score_loss(y, p_fnn)),
    glm_lift10=float(top_decile_capture(y, p_glm)),
    fnn_lift10=float(top_decile_capture(y, p_fnn)),
)

# Severity metrics
rmse_log_glm = float(np.sqrt(np.mean((np.log1p(y_sev)-np.log1p(sev_glm))**2)))
rmse_log_fnn = float(np.sqrt(np.mean((np.log1p(y_sev)-np.log1p(sev_fnn))**2)))
q90 = np.quantile(y_sev, 0.90)
tail = y_sev >= q90
tail_mae_glm = float(mean_absolute_error(y_sev[tail], sev_glm[tail]))
tail_mae_fnn = float(mean_absolute_error(y_sev[tail], sev_fnn[tail]))

# Aggregate
exp_months = float(glm_expo["exposure_months"].sum())
act_total = float(y_sev.sum())
pp_act = act_total / exp_months

exp_glm = float(glm_expo["exp_cost"].sum())
exp_fnn = float(fnn_expo["exp_cost"].sum())

pp_glm = exp_glm / exp_months
pp_fnn = exp_fnn / exp_months
oe_glm = act_total / max(exp_glm, 1e-12)
oe_fnn = act_total / max(exp_fnn, 1e-12)

# Hybrids
pp_fnnInc_glmSev = float(np.sum(p_fnn * glm_expo["sev_hat"].values.astype(float))) / exp_months
pp_glmInc_fnnSev = float(np.sum(p_glm * fnn_expo["sev_hat"].values.astype(float))) / exp_months

comparison = pd.DataFrame([
    {"Layer":"Incidence","Metric":"Observed rate","Actual":inc["observed_rate"],"GLM":inc["glm_mean_p"],"FFNN":inc["fnn_mean_p"],"Notes":"Calibration-in-the-large"},
    {"Layer":"Incidence","Metric":"AUC (ROC)","Actual":np.nan,"GLM":inc["glm_auc"],"FFNN":inc["fnn_auc"],"Notes":"Ranking quality"},
    {"Layer":"Incidence","Metric":"PR-AUC","Actual":np.nan,"GLM":inc["glm_pr"],"FFNN":inc["fnn_pr"],"Notes":"Rare-event ranking"},
    {"Layer":"Incidence","Metric":"Brier (↓)","Actual":np.nan,"GLM":inc["glm_brier"],"FFNN":inc["fnn_brier"],"Notes":"Probability accuracy"},
    {"Layer":"Incidence","Metric":"Lift@10","Actual":np.nan,"GLM":inc["glm_lift10"],"FFNN":inc["fnn_lift10"],"Notes":"Claims captured in top 10%"},
    {"Layer":"Claim cost","Metric":"Mean cost (CHF)","Actual":float(y_sev.mean()),"GLM":float(sev_glm.mean()),"FFNN":float(sev_fnn.mean()),"Notes":"Level check (claims)"},
    {"Layer":"Claim cost","Metric":"MAE (CHF) (↓)","Actual":np.nan,"GLM":float(mean_absolute_error(y_sev, sev_glm)),"FFNN":float(mean_absolute_error(y_sev, sev_fnn)),"Notes":"Overall error"},
    {"Layer":"Claim cost","Metric":"RMSE log1p (↓)","Actual":np.nan,"GLM":rmse_log_glm,"FFNN":rmse_log_fnn,"Notes":"Multiplicative error"},
    {"Layer":"Claim cost","Metric":"Tail MAE P90+ (↓)","Actual":np.nan,"GLM":tail_mae_glm,"FFNN":tail_mae_fnn,"Notes":"Large-loss focus"},
    {"Layer":"Aggregate","Metric":"Pure premium (CHF/exp-month)","Actual":pp_act,"GLM":pp_glm,"FFNN":pp_fnn,"Notes":"p×mu over exposure"},
    {"Layer":"Aggregate","Metric":"O/E (Actual ÷ Expected)","Actual":1.0,"GLM":oe_glm,"FFNN":oe_fnn,"Notes":"Ideal ≈ 1"},
    {"Layer":"Diagnostics","Metric":"PP (FFNN incidence × GLM severity)","Actual":pp_act,"GLM":pp_glm,"FFNN":pp_fnnInc_glmSev,"Notes":"Isolate frequency"},
    {"Layer":"Diagnostics","Metric":"PP (GLM incidence × FFNN severity)","Actual":pp_act,"GLM":pp_glm,"FFNN":pp_glmInc_fnnSev,"Notes":"Isolate severity"},
])

comparison["FFNN/GLM"] = comparison["FFNN"] / comparison["GLM"]
comparison


,Layer,Metric,Actual,GLM,FFNN,Notes,FFNN/GLM
0,Incidence,Observed rate,0.010667,0.012385,0.011431,Calibration-in-the-large,0.922980
1,Incidence,AUC (ROC),NaN,0.606511,0.624689,Ranking quality,1.029971
2,Incidence,PR-AUC,NaN,0.016580,0.018605,Rare-event ranking,1.122168
3,Incidence,Brier (↓),NaN,0.010627,0.010542,Probability accuracy,0.991962
4,Incidence,Lift@10,NaN,0.213542,0.247396,Claims captured in top 10%,1.158537
5,Claim cost,Mean cost (CHF),106135.330729,106932.363968,103041.750167,Level check (claims),0.963616
6,Claim cost,MAE (CHF) (↓),NaN,47339.382741,36608.375287,Overall error,0.773318
7,Claim cost,RMSE log1p (↓),NaN,0.558704,0.431314,Multiplicative error,0.771990
8,Claim cost,Tail MAE P90+ (↓),NaN,89129.645953,108032.198208,Large-loss focus,1.212079
9,Aggregate,Pure premium (CHF/exp-month),1132.110194,1310.617736,1181.918907,p×mu over exposure,0.901803


In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, mean_absolute_error

def top_decile_capture(y_true, p_hat, frac=0.10):
    k = max(1, int(frac * len(p_hat)))
    idx = np.argsort(p_hat)[-k:]
    return float(y_true[idx].sum() / max(1, y_true.sum()))

# --- Inputs from artifacts (already loaded in Notebook 05) ---
# glm_expo, glm_clm, fnn_expo, fnn_clm

# Exposure-level arrays (same test window)
y_inc   = glm_expo["onset"].values.astype(float)
p_glm   = glm_expo["p_hat"].values.astype(float)
p_fnn   = fnn_expo["p_hat"].values.astype(float)

mu_glm_expo = glm_expo["sev_hat"].values.astype(float)   # GLM severity scored on exposure rows
mu_fnn_expo = fnn_expo["sev_hat"].values.astype(float)   # FFNN severity scored on exposure rows

exp_months = float(glm_expo["exposure_months"].sum())

# Claims-level arrays (severity evaluation on claims)
y_sev = glm_clm["total_paid_ultimate"].values.astype(float)
mu_glm_claims = glm_clm["sev_hat"].values.astype(float)
mu_fnn_claims = fnn_clm["sev_hat"].values.astype(float)

# --- Incidence metrics ---
inc_glm = {
    "Observed rate": float(y_inc.mean()),
    "AUC (ROC)": float(roc_auc_score(y_inc, p_glm)),
    "PR-AUC": float(average_precision_score(y_inc, p_glm)),
    "Brier (↓)": float(brier_score_loss(y_inc, p_glm)),
    "Lift@10": float(top_decile_capture(y_inc, p_glm)),
    "Mean p": float(p_glm.mean()),
}
inc_fnn = {
    "Observed rate": float(y_inc.mean()),
    "AUC (ROC)": float(roc_auc_score(y_inc, p_fnn)),
    "PR-AUC": float(average_precision_score(y_inc, p_fnn)),
    "Brier (↓)": float(brier_score_loss(y_inc, p_fnn)),
    "Lift@10": float(top_decile_capture(y_inc, p_fnn)),
    "Mean p": float(p_fnn.mean()),
}

# Hybrid ownership for incidence:
# Hybrid A uses FFNN incidence; Hybrid B uses GLM incidence
inc_hA = inc_fnn
inc_hB = inc_glm

# --- Severity metrics (on claims) ---
rmse_log_glm = float(np.sqrt(np.mean((np.log1p(y_sev) - np.log1p(mu_glm_claims))**2)))
rmse_log_fnn = float(np.sqrt(np.mean((np.log1p(y_sev) - np.log1p(mu_fnn_claims))**2)))

q90 = np.quantile(y_sev, 0.90)
tail = y_sev >= q90

sev_glm = {
    "Mean cost (CHF)": float(y_sev.mean()),
    "Mean pred (CHF)": float(mu_glm_claims.mean()),
    "MAE (CHF) (↓)": float(mean_absolute_error(y_sev, mu_glm_claims)),
    "RMSE log1p (↓)": rmse_log_glm,
    "Tail MAE P90+ (↓)": float(mean_absolute_error(y_sev[tail], mu_glm_claims[tail])),
}
sev_fnn = {
    "Mean cost (CHF)": float(y_sev.mean()),
    "Mean pred (CHF)": float(mu_fnn_claims.mean()),
    "MAE (CHF) (↓)": float(mean_absolute_error(y_sev, mu_fnn_claims)),
    "RMSE log1p (↓)": rmse_log_fnn,
    "Tail MAE P90+ (↓)": float(mean_absolute_error(y_sev[tail], mu_fnn_claims[tail])),
}

# Hybrid ownership for severity:
# Hybrid A uses GLM severity; Hybrid B uses FFNN severity
sev_hA = sev_glm
sev_hB = sev_fnn

# --- Aggregate totals and PP/OE ---
actual_total = float(y_sev.sum())  # actual total cost in test claims
pp_act = actual_total / exp_months

# Expected totals:
exp_total_glm = float(glm_expo["exp_cost"].sum())  # = sum(p_glm * mu_glm_expo)
exp_total_fnn = float(fnn_expo["exp_cost"].sum())  # = sum(p_fnn * mu_fnn_expo)

# Hybrids:
exp_total_hA = float(np.sum(p_fnn * mu_glm_expo))  # FFNN incidence × GLM severity
exp_total_hB = float(np.sum(p_glm * mu_fnn_expo))  # GLM incidence × FFNN severity

def agg_stats(exp_total):
    pp = exp_total / exp_months
    oe = actual_total / max(exp_total, 1e-12)
    return pp, oe

pp_glm, oe_glm = agg_stats(exp_total_glm)
pp_fnn, oe_fnn = agg_stats(exp_total_fnn)
pp_hA,  oe_hA  = agg_stats(exp_total_hA)
pp_hB,  oe_hB  = agg_stats(exp_total_hB)

# --- Build expanded comparison table ---
rows = [
    # Incidence
    ("Incidence", "Observed rate", inc_glm["Observed rate"], inc_glm["Mean p"], inc_fnn["Mean p"], inc_hA["Mean p"], inc_hB["Mean p"],
     "Calibration-in-the-large"),
    ("Incidence", "AUC (ROC)", np.nan, inc_glm["AUC (ROC)"], inc_fnn["AUC (ROC)"], inc_hA["AUC (ROC)"], inc_hB["AUC (ROC)"],
     "Ranking quality"),
    ("Incidence", "PR-AUC", np.nan, inc_glm["PR-AUC"], inc_fnn["PR-AUC"], inc_hA["PR-AUC"], inc_hB["PR-AUC"],
     "Rare-event ranking"),
    ("Incidence", "Brier (↓)", np.nan, inc_glm["Brier (↓)"], inc_fnn["Brier (↓)"], inc_hA["Brier (↓)"], inc_hB["Brier (↓)"],
     "Probability accuracy"),
    ("Incidence", "Lift@10", np.nan, inc_glm["Lift@10"], inc_fnn["Lift@10"], inc_hA["Lift@10"], inc_hB["Lift@10"],
     "Claims captured in top 10%"),

    # Severity (on claims)
    ("Claim cost", "Mean cost (CHF)", sev_glm["Mean cost (CHF)"], sev_glm["Mean pred (CHF)"], sev_fnn["Mean pred (CHF)"],
     sev_hA["Mean pred (CHF)"], sev_hB["Mean pred (CHF)"], "Level check (claims)"),
    ("Claim cost", "MAE (CHF) (↓)", np.nan, sev_glm["MAE (CHF) (↓)"], sev_fnn["MAE (CHF) (↓)"],
     sev_hA["MAE (CHF) (↓)"], sev_hB["MAE (CHF) (↓)"], "Overall error"),
    ("Claim cost", "RMSE log1p (↓)", np.nan, sev_glm["RMSE log1p (↓)"], sev_fnn["RMSE log1p (↓)"],
     sev_hA["RMSE log1p (↓)"], sev_hB["RMSE log1p (↓)"], "Multiplicative error"),
    ("Claim cost", "Tail MAE P90+ (↓)", np.nan, sev_glm["Tail MAE P90+ (↓)"], sev_fnn["Tail MAE P90+ (↓)"],
     sev_hA["Tail MAE P90+ (↓)"], sev_hB["Tail MAE P90+ (↓)"], "Large-loss focus"),

    # Aggregate
    ("Aggregate", "Pure premium (CHF/exp-month)", pp_act, pp_glm, pp_fnn, pp_hA, pp_hB, "p×mu over exposure"),
    ("Aggregate", "O/E (Actual ÷ Expected)", 1.0, oe_glm, oe_fnn, oe_hA, oe_hB, "Ideal ≈ 1"),
]

comparison = pd.DataFrame(rows, columns=[
    "Layer","Metric","Actual","GLM","FFNN","Hybrid A","Hybrid B","Notes"
])

# Ratios (optional but useful)
comparison["FFNN/GLM"]     = comparison["FFNN"] / comparison["GLM"]
comparison["HybridA/GLM"]  = comparison["Hybrid A"] / comparison["GLM"]
comparison["HybridB/GLM"]  = comparison["Hybrid B"] / comparison["GLM"]

display(comparison)

# wrtie to csv
OUT_DIR = "outputs_slide_assets"
os.makedirs(OUT_DIR, exist_ok=True)
comparison.to_csv(os.path.join(OUT_DIR, "model_comparison_scorecard.csv"), index=False)

,Layer,Metric,Actual,GLM,FFNN,Hybrid A,Hybrid B,Notes,FFNN/GLM,HybridA/GLM,HybridB/GLM
0,Incidence,Observed rate,0.010667,0.012385,0.011431,0.011431,0.012385,Calibration-in-the-large,0.922980,0.922980,1.000000
1,Incidence,AUC (ROC),NaN,0.606511,0.624689,0.624689,0.606511,Ranking quality,1.029971,1.029971,1.000000
2,Incidence,PR-AUC,NaN,0.016580,0.018605,0.018605,0.016580,Rare-event ranking,1.122168,1.122168,1.000000
3,Incidence,Brier (↓),NaN,0.010627,0.010542,0.010542,0.010627,Probability accuracy,0.991962,0.991962,1.000000
4,Incidence,Lift@10,NaN,0.213542,0.247396,0.247396,0.213542,Claims captured in top 10%,1.158537,1.158537,1.000000
5,Claim cost,Mean cost (CHF),106135.330729,106932.363968,103041.750167,106932.363968,103041.750167,Level check (claims),0.963616,1.000000,0.963616
6,Claim cost,MAE (CHF) (↓),NaN,47339.382741,36608.375287,47339.382741,36608.375287,Overall error,0.773318,1.000000,0.773318
7,Claim cost,RMSE log1p (↓),NaN,0.558704,0.431314,0.558704,0.431314,Multiplicative error,0.771990,1.000000,0.771990
8,Claim cost,Tail MAE P90+ (↓),NaN,89129.645953,108032.198208,89129.645953,108032.198208,Large-loss focus,1.212079,1.000000,1.212079
9,Aggregate,Pure premium (CHF/exp-month),1132.110194,1310.617736,1181.918907,1200.227482,1280.652694,p×mu over exposure,0.901803,0.915772,0.977137


In [11]:
ARTIFACT_DIR = "outputs_artifacts"

glm_expo = pd.read_csv(f"{ARTIFACT_DIR}/glm_test_exposure_scored.csv", parse_dates=["month_start"])
glm_clm  = pd.read_csv(f"{ARTIFACT_DIR}/glm_test_claims_scored.csv",   parse_dates=["month_start"])
fnn_expo = pd.read_csv(f"{ARTIFACT_DIR}/ffnn_test_exposure_scored.csv", parse_dates=["month_start"])
fnn_clm  = pd.read_csv(f"{ARTIFACT_DIR}/ffnn_test_claims_scored.csv",   parse_dates=["month_start"])

exp_months = float(glm_expo["exposure_months"].sum())
actual_total = float(glm_clm["total_paid_ultimate"].sum())  # same test window for both models

expected_total_glm  = float(glm_expo["exp_cost"].sum())
expected_total_fnn  = float(fnn_expo["exp_cost"].sum())

p_glm = glm_expo["p_hat"].values
p_fnn = fnn_expo["p_hat"].values
sev_glm = glm_expo["sev_hat"].values
sev_fnn = fnn_expo["sev_hat"].values

expected_total_hybrid_A = float(np.sum(p_fnn * sev_glm))  # FFNN incidence × GLM severity
expected_total_hybrid_B = float(np.sum(p_glm * sev_fnn))  # GLM incidence × FFNN severity

print("Exposure months:", f"{exp_months:,.1f}")
print("Actual total cost (CHF):", f"{actual_total:,.0f}")
print("Expected total cost GLM (CHF):", f"{expected_total_glm:,.0f}")
print("Expected total cost FFNN (CHF):", f"{expected_total_fnn:,.0f}")
print("Expected total Hybrid A (CHF):", f"{expected_total_hybrid_A:,.0f}")
print("Expected total Hybrid B (CHF):", f"{expected_total_hybrid_B:,.0f}")

totals = pd.DataFrame([
    {"Model": "Actual", "Expected Total (CHF)": f"{actual_total:,.0f}"},
    {"Model": "GLM", "Expected Total (CHF)": f"{expected_total_glm:,.0f}"},
    {"Model": "FFNN", "Expected Total (CHF)": f"{expected_total_fnn:,.0f}"},
    {"Model": "Hybrid A (FFNN incidence × GLM severity)", "Expected Total (CHF)": f"{expected_total_hybrid_A:,.0f}"},
    {"Model": "Hybrid B (GLM incidence × FFNN severity)", "Expected Total (CHF)": f"{expected_total_hybrid_B:,.0f}"},
])

# export to csv
totals.to_csv(os.path.join(ARTIFACT_DIR, "expected_totals_comparison.csv"), index=False)

Exposure months: 36,000.0
Actual total cost (CHF): 40,755,967
Expected total cost GLM (CHF): 47,182,238
Expected total cost FFNN (CHF): 42,549,081
Expected total Hybrid A (CHF): 43,208,189
Expected total Hybrid B (CHF): 46,103,497


In [12]:
# Save scorecard
out_csv = os.path.join(ARTIFACT_DIR, "comparison_scorecard_from_artifacts.csv")
comparison.to_csv(out_csv, index=False)
out_csv

'outputs_artifacts\\comparison_scorecard_from_artifacts.csv'